<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/Portfolio_Construction_(Concentration).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. Negative Correlation

Value and Momentum are the Yin and Yang of quantitative finance.

* Value is contrarian: You are buying companies that everyone hates, waiting for them to revert to the mean.

* Momentum is trend-following: You are buying companies that everyone loves, betting the trend will persist.

Because they rely on completely opposite behavioral biases, their periods of underperformance rarely overlap. When Momentum crashes (like in 2009), Value usually skyrockets. When Value is dead (like the late 2010s tech boom), Momentum prints money.

### B. The Blending Math (Z-Scores)

How do you combine a P/E ratio of 8.5 with a Momentum return of +45%? You can't add them; the units are different.
Quants solve this using Cross-Sectional Z-Scores.

1. We calculate the Mean and Standard Deviation of the Value metric across our universe.

2. We convert every stock's Value metric into a Z-Score (how many standard deviations it is above or below the mean).

3. We do the exact same thing for Momentum.

4. We add the two Z-Scores together. A stock with a high combined Z-Score is both fundamentally cheap AND behaviorally trending.

### Multi-Factor Z-Score Engine

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [2]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL', # Big Tech
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',                    # Financials
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',                       # Energy
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',                      # Healthcare
    'PG', 'KO', 'PEP', 'WMT', 'TGT',                         # Staples/Retail
    'CAT', 'DE', 'GE', 'HON', 'MMM'                          # Industrials
]

In [3]:
# We need 252 trading days (1 year) + extra buffer for the rolling calculations
data = yf.download(universe, period="18mo")['Close']
data = data.dropna(axis=1) # Drop any stocks with missing data


[*********************100%***********************]  33 of 33 completed


In [5]:

# Momentum metrics

# Constants for trading days
TRADING_DAYS_12M = 252
TRADING_DAYS_1M = 21

# Metric 1: The Naive 12-Month Return
# Formula: (Price Today / Price 252 days ago) - 1
naive_12m_return = data.pct_change(TRADING_DAYS_12M)

# Metric 2: The Academic 12m-1m Return
# Formula: (Price 21 days ago / Price 252 days ago) - 1
# We use .shift() to look backward in time
price_t_minus_1m = data.shift(TRADING_DAYS_1M)
price_t_minus_12m = data.shift(TRADING_DAYS_12M)

academic_12m_1m_return = (price_t_minus_1m / price_t_minus_12m) - 1
# Get the most recent valid day
latest_date = academic_12m_1m_return.dropna().index[-1]

In [6]:
# We will use yfinance to fetch trailing P/E as our fundamental value metric
value_data = []
for ticker in universe:
    try:
        # Fetching fundamental info
        stock = yf.Ticker(ticker)
        pe_ratio = stock.info.get('trailingPE', np.nan)

        # We want Earnings Yield (E/P), so higher is better (cheaper)
        # If PE is negative or missing, we assign NaN
        earnings_yield = (1 / pe_ratio) if pd.notna(pe_ratio) and pe_ratio > 0 else np.nan

        value_data.append({'Ticker': ticker, 'Earnings_Yield': earnings_yield})
    except:
        value_data.append({'Ticker': ticker, 'Earnings_Yield': np.nan})

value_df = pd.DataFrame(value_data).set_index('Ticker')

# Fill missing fundamental data with the universe median to prevent dropping stocks
median_ey = value_df['Earnings_Yield'].median()
value_df['Earnings_Yield'] = value_df['Earnings_Yield'].fillna(median_ey)

In [10]:
# Retrieve the Momentum metric
momentum_series = academic_12m_1m_return.loc[latest_date].dropna()

# Align the dataframes perfectly
factor_df = pd.DataFrame({
    'Momentum': momentum_series,
    'Value': value_df['Earnings_Yield']
}).dropna()

# Calculate Z-Scores
# Z = (Value - Mean) / Standard Deviation
factor_df['Mom_Z'] = (factor_df['Momentum'] - factor_df['Momentum'].mean()) / factor_df['Momentum'].std()
factor_df['Val_Z'] = (factor_df['Value'] - factor_df['Value'].mean()) / factor_df['Value'].std()



In [11]:
# The Composite Score is the equal-weighted sum of the two Z-scores
factor_df['Composite_Z'] = factor_df['Mom_Z'] + factor_df['Val_Z']

# Sort to find the highest combined score
factor_df = factor_df.sort_values('Composite_Z', ascending=False)

In [12]:
print("\n" + "="*80)
print(" MULTI-FACTOR RANKING: VALUE + MOMENTUM")
print("="*80)
print(f"{'Ticker':<10} | {'Mom Z-Score':<15} | {'Val Z-Score':<15} | {'Composite Z'}")
print("-" * 80)

for ticker, row in factor_df.head(10).iterrows():
    print(f"{ticker:<10} | {row['Mom_Z']:>11.2f}     | {row['Val_Z']:>11.2f}     | {row['Composite_Z']:>9.2f}")
print("="*80)

# Identify the "Pure" winners vs the "Blended" winners
pure_mom_winner = factor_df['Mom_Z'].idxmax()
pure_val_winner = factor_df['Val_Z'].idxmax()
composite_winner = factor_df.index[0]

print(f"\n📊 PORTFOLIO INSIGHTS:")
print(f"Top Pure Momentum Stock:  {pure_mom_winner}")
print(f"Top Pure Value Stock:     {pure_val_winner}")
if composite_winner not in [pure_mom_winner, pure_val_winner]:
    print(f"Top Composite Stock:      {composite_winner} 🏆")
    print(f"Notice how {composite_winner} isn't the absolute cheapest, nor the absolute highest momentum,")
    print("but it possesses the best mathematical combination of both factors.")
else:
    print(f"Top Composite Stock:      {composite_winner} 🏆")
    print(f"Remarkably, {composite_winner} dominates the composite score by being an extreme outlier in one or both factors.")


 MULTI-FACTOR RANKING: VALUE + MOMENTUM
Ticker     | Mom Z-Score     | Val Z-Score     | Composite Z
--------------------------------------------------------------------------------
GOOGL      |        2.87     |       -0.16     |      2.71
CAT        |        3.17     |       -1.18     |      1.99
C          |        0.80     |        0.89     |      1.70
SLB        |        1.16     |        0.46     |      1.62
EOG        |       -0.40     |        2.00     |      1.61
GS         |        0.67     |        0.58     |      1.24
WFC        |       -0.94     |        2.07     |      1.14
MS         |        0.58     |        0.54     |      1.13
BAC        |       -0.58     |        1.67     |      1.10
COP        |        0.06     |        0.83     |      0.89

📊 PORTFOLIO INSIGHTS:
Top Pure Momentum Stock:  CAT
Top Pure Value Stock:     WFC
Top Composite Stock:      GOOGL 🏆
Notice how GOOGL isn't the absolute cheapest, nor the absolute highest momentum,
but it possesses the best mat